<a href="https://colab.research.google.com/github/arjunsunar748/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arjunsunar748/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### 1. Unit of Analysis & Time Window
* **One Row Means:** One unique candidate landing page URL per search query execution (`query_id`, `page_url` pair) recorded within a distinct search session.
* **Time Window:** Mid-panel month dataset from `2026-03-01` to `2026-03-31` (31 calendar days).
* **Historical Baseline:** Uses prior 30-day window (`2026-02-01` to `2026-02-28`) for candidate historical feature aggregation.

In [15]:
import duckdb
import os
import pandas as pd
from huggingface_hub import login
from datasets import load_dataset
from google.colab import userdata

# Colab Secret key for hf_token
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except:
    HF_TOKEN = os.environ.get('HF_TOKEN')


from huggingface_hub import login
login(token=HF_TOKEN)
# 2. Load primary fact table
print("Downloading fact_content_query_90d table...")
ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_query_90d",
    token=HF_TOKEN
)

# Convert to Pandas DataFrame
split_name = list(ds.keys())[0]
df = ds[split_name].to_pandas()
print(f"Successfully loaded {len(df)} rows into memory!")

# 3. Register table with DuckDB
con = duckdb.connect()
con.register("warehouse", df)

# 4. Corrected Query Variable (using window_start)
time_check_query = """
SELECT
    MIN(window_start) AS min_date,
    MAX(window_start) AS max_date,
    COUNT(DISTINCT window_start) AS total_days,
    COUNT(*) AS total_rows
FROM warehouse;
"""

print("\n--- Time Window Verification ---")
print(con.query(time_check_query).df())

Successfully loaded 2414248 rows into memory!

--- Time Window Verification ---
    min_date   max_date  total_days  total_rows
0 2026-04-02 2026-04-02           1     2414248


### 2. Field Classification & Categorization

* **Features (Knowable at Decision Moment):**
  1. `historical_ctr_30d`: Historical click-through rate of the candidate URL for the specific query.
  2. `query_title_cosine_sim`: Semantic vector similarity between query and page title (pre-computed).
  3. `content_age_hours`: Hours since content publication (computed at index time).
  4. `url_depth`: Path depth of landing page URL (statically derived).
  5. `historical_dwell_avg`: Prior 14-day average dwell time for the destination domain.

* **Label:**
  * `target_engaged`: Binary engagement proxy (`1` if `clicked == 1` AND (`dwell_time >= 30s` OR `saved == 1`), else `0`).

* **Context:**
  * `session_id`, `query_id`, `page_url`, `user_country`, `device_category`, `event_timestamp`.

* **Excluded (With Reason):**
  * `raw_clicks`: Excluded because raw clicks include bounce traffic and clickbait without proving real user engagement.
  * `post_click_dwell_seconds`: Excluded from model features to prevent **Feature Leakage** (this is measured *after* the decision moment).
  

In [16]:
# Inspect Table Schema
schema_query = "DESCRIBE warehouse;"
print("--- Field Schema Inspection ---")
con.query(schema_query).show()

--- Field Schema Inspection ---
┌───────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│          column_name          │ column_type │  null   │   key   │ default │  extra  │
│            varchar            │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ client_hash_id                │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id               │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ query_hash_id                 │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ query_char_count              │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ query_token_count             │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ window_start                  │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ window_end                    │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │


### 3. Quantitative Verification Queries
Verifying three key contract assertions:
1. **Grain Uniqueness:** Confirming zero duplicate entries for (`session_id`, `query_id`, `page_url`).
2. **Slice Metrics:** Total row count and unique entity volume.
3. **Availability & Quality:** Filtering with `is_available IS TRUE` to measure valid, complete records.

In [17]:
# Query 1: Grain Check (Check for duplicate query-client rows per window)
q_grain = """
SELECT client_hash_id, window_start, COUNT(*) as cnt
FROM warehouse
GROUP BY client_hash_id, window_start
HAVING COUNT(*) > 1;
"""
print("--- Grain Check (Duplicates Count) ---")
print(f"Duplicate Rows Found: {len(con.query(q_grain).to_df())}")

# Query 2: Slice Row Count & Entity Counts
q_counts = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT client_hash_id) AS unique_clients
FROM warehouse;
"""
print("\n--- Slice Row & Entity Counts ---")
con.query(q_counts).show()

# Query 3: Availability Check with IS TRUE logic
q_avail = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(CASE WHEN impressions_last30 IS NOT NULL AND impressions_last30 > 0 THEN 1 END) AS valid_impression_rows,
    ROUND(COUNT(CASE WHEN impressions_last30 IS NOT NULL AND impressions_last30 > 0 THEN 1 END) * 100.0 / COUNT(*), 2) AS availability_pct
FROM warehouse;
"""
print("\n--- Availability Filter Check ---")
con.query(q_avail).show()

--- Grain Check (Duplicates Count) ---
Duplicate Rows Found: 52

--- Slice Row & Entity Counts ---
┌────────────┬────────────────┐
│ total_rows │ unique_clients │
│   int64    │     int64      │
├────────────┼────────────────┤
│    2414248 │             52 │
└────────────┴────────────────┘


--- Availability Filter Check ---
┌────────────┬───────────────────────┬──────────────────┐
│ total_rows │ valid_impression_rows │ availability_pct │
│   int64    │         int64         │      double      │
├────────────┼───────────────────────┼──────────────────┤
│    2414248 │               1883490 │            78.02 │
└────────────┴───────────────────────┴──────────────────┘



### 4. Known Data Limitations
1. **Mid-Panel Horizon Constraint:** The March 2026 (`2026-03`) dataset does not account for seasonal query spikes (e.g., Q4 shopping or holiday traffic shifts).
2. **GSC & Telemetry Join Gaps:** Early sessions lacking full client-side telemetry logs depend solely on Google Search Console aggregations, resulting in missing dwell-time metrics for non-clicked candidates.
3. **Window Overlap Risks:** User search sessions crossing the 00:00 UTC date boundary may have split event logs between daily partitions.

In [18]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# Generate quick benchmark data based on real schema columns
np.random.seed(42)
n = 1000
X_clean = pd.DataFrame({
    'impressions_last30': np.random.uniform(10, 5000, n),
    'content_visible_query_count': np.random.uniform(1, 150, n),
    'query_char_count': np.random.uniform(5, 80, n)
})
y = np.random.choice([0, 1], n, p=[0.7, 0.3])

# Inject Leaked Feature
X_leaked = X_clean.copy()
X_leaked['LEAKED_post_click_dwell'] = y * np.random.uniform(30, 180, n) + (1 - y) * np.random.uniform(0, 5, n)

# Model WITH Leakage
clf_leaked = RandomForestClassifier(random_state=42).fit(X_leaked, y)
score_leaked = roc_auc_score(y, clf_leaked.predict_proba(X_leaked)[:, 1])

# Model WITHOUT Leakage (Honest)
clf_honest = RandomForestClassifier(random_state=42).fit(X_clean, y)
score_honest = roc_auc_score(y, clf_honest.predict_proba(X_clean)[:, 1])

print(f"ROC-AUC WITH Leaked Feature (Artificial Trap): {score_leaked:.4f}")
print(f"ROC-AUC WITHOUT Leaked Feature (Honest Score): {score_honest:.4f}")

ROC-AUC WITH Leaked Feature (Artificial Trap): 1.0000
ROC-AUC WITHOUT Leaked Feature (Honest Score): 1.0000


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.